In [1]:
"""
ReAct Agent Example: Trip Cost Estimator
Pattern: Reasoning + Acting in a loop
 
The agent reasons about what it needs, decides to call tools (functions),
executes those tools, observes the results, and loops until it has an answer.
 
Structure:
  1. Configuration (LL  M setup)
  2. Tool definitions (functions the agent can call)
  3. Agent class (maintains conversation history)
  4. Action parsing (extract tool calls from LLM output)
  5. ReAct loop (the main agent loop)
  6. Usage examples
"""
import os
import re
from openai import OpenAI
from dotenv import load_dotenv
from openai import OpenAI
from typing import Optional, Any
from pydantic import BaseModel

In [2]:
class LLMSetup(BaseModel):
    llm_model: str = "gpt-4"
    llm_client: Any


def load_llm() -> LLMSetup:
    api_key = os.getenv("OPENAI_API_KEY")

    if not api_key:
        raise EnvironmentError("OPENAI_API_KEY not found in env")

    llm_client = OpenAI(api_key=api_key)
    return LLMSetup(llm_client=llm_client)

In [3]:
def estimate_trip_cost(days: int, travelers: int, comfort: str = "mid") -> str:
    """
    Estimate a trip budget in SGD using simple heuristics.

    This is a TOOL that the agent can call. The agent provides:
    - days: number of days for the trip
    - travelers: number of people traveling
    - comfort: budget level (budget | mid | premium)

    Returns:
    - A string with the total estimated cost

    The agent uses this tool when it needs to know the cost of a trip.
    """
    # Validate inputs
    if days <= 0 or travelers <= 0:
        raise ValueError("days and travelers must be > 0")

    comfort = comfort.lower().strip()
    if comfort not in {"budget", "mid", "premium"}:
        raise ValueError("comfort must be one of: budget, mid, premium")

    # Cost estimates per person per day in SGD
    # These are rough heuristics, not real-world data
    daily_costs = {
        "budget": {
            "lodging": 60,
            "food": 30,
            "transport": 10,
            "activities": 20,
        },
        "mid": {
            "lodging": 140,
            "food": 60,
            "transport": 20,
            "activities": 50,
        },
        "premium": {
            "lodging": 300,
            "food": 120,
            "transport": 50,
            "activities": 120,
        },
    }

    # Calculate costs for all travelers over all days
    costs = daily_costs[comfort]
    lodging = costs["lodging"] * travelers * days
    food = costs["food"] * travelers * days
    transport = costs["transport"] * travelers * days
    activities = costs["activities"] * travelers * days

    subtotal = lodging + food + transport + activities
    # Add 12% buffer for contingencies
    contingency = round(subtotal * 0.12)
    total = subtotal + contingency

    # Return in a format the agent can parse
    return f"total_estimate: {total}"


In [ ]:
class ReActAgent:
    """
    A ReAct (Reasoning + Acting) agent that maintains conversation history
    and calls the LLM to reason about a problem and decide what actions to take.

    Key insight: The agent keeps ALL messages (system, user, assistant) in
    self.messages. This way, the LLM can see the entire conversation history
    and make decisions based on what has already been discussed.
    """

    def __init__(self, system_prompt: str, llm_setup: LLMSetup):
        """
        Initialize the agent with a system prompt.

        Args:
            system_prompt: Instructions for the agent (e.g., "You run in a
                          Thought-Action-Observation loop...")
        """
        self.llm = llm_setup
        self.messages = []
        if system_prompt:
            # Add system prompt as the first message
            # This tells the LLM how to behave
            self.messages.append({"role": "system", "content": system_prompt})

    def __call__(self, user_message: str) -> str:
        """
        Send a message to the agent and get a response.

        This method:
        1. Adds the user message to history
        2. Calls the LLM
        3. Adds the LLM response to history
        4. Returns the response

        Args:
            user_message: What the user or agent loop sends to the LLM

        Returns:
            The LLM's response (may contain Action, Thought, or Answer)
        """
        # Add user message to conversation history
        self.messages.append({"role": "user", "content": user_message})

        # Call the LLM with the full conversation history
        response = self._call_llm()

        # Add LLM response to conversation history
        # This is crucial: the next turn, the LLM will see this response
        self.messages.append({"role": "assistant", "content": response})

        return response

    def _call_llm(self) -> str:
        """
        Make an API call to the LLM with the current message history.

        The LLM sees all messages (system + entire conversation history),
        which allows it to understand context and make informed decisions.

        Returns:
            The LLM's response as a string
        """
        response = self.llm.llm_client.chat.completions.create(
            model=self.llm.llm_model,
            temperature=0.4,  # Lower temperature = more focused, deterministic
            messages=self.messages,
        )
        return response.choices[0].message.content


In [5]:
llm_client = load_llm()
# Dictionary of all available tools
# The agent can only call tools that are in this dictionary
AVAILABLE_TOOLS = {"estimate_trip_cost": estimate_trip_cost}
